# 03 — Feature Engineering

This notebook converts preprocessed text into numeric representations (BoW, TF-IDF, embeddings) and computes pairwise similarity features (cosine, Euclidean, Manhattan).

In [5]:
import sys
from pathlib import Path

import pandas as pd

%load_ext autoreload
%autoreload 2

def _find_project_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src").exists() and (p / "data").exists():
            return p
    return start.parent

PROJECT_ROOT = _find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_PATH))

from feature_engineering import (
    build_bow,
    build_tfidf,
    cosine_sim_sparse,
    l2_distance_sparse,
    l1_distance_sparse,
    cosine_sim_dense,
    l2_distance_dense,
    l1_distance_dense,
    build_glove_avg,
    build_word2vec_avg,
    build_bert_embeddings,
    build_similarity_frame,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import importlib.util

need = []
if importlib.util.find_spec("gensim") is None:
    need.append("gensim")
if importlib.util.find_spec("sentence_transformers") is None:
    need.append("sentence-transformers")

if need:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + need)

In [7]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
parquet_path = PROCESSED_DIR / "quora_preprocessed.parquet"
csv_path = PROCESSED_DIR / "quora_preprocessed.csv"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
elif csv_path.exists():
    df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError("Run 02_preprocessing.ipynb first to create quora_preprocessed.(parquet|csv)")

cols = ["id", "q1_classic", "q2_classic", "q1_norm", "q2_norm"]
missing = [c for c in cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns in preprocessed dataset: {missing}")

df[cols].head()

,id,q1_classic,q2_classic,q1_norm,q2_norm
0,332278,iliad odyssey greek culture,prove pairs three independent variables also i...,the iliad and the odyssey in the greek culture,how do i prove that the pairs of three indepen...
1,196656,practical management strategic management,practical aspects strategic management,what is practical management and what is strat...,what are the practical aspects of strategic ma...
2,113125,useful makeuseof answers,q site yahoo answers hate speech allowed,how useful is makeuseof answers,is there any q a site that is not yahoo answer...
3,266232,best place reside india,ia best place visit india,which is the best place to reside in india and...,which ia the best place to visit in india
4,122738,many people ask questions quora easily answere...,many people posting questions quora check goog...,why do so many people ask questions on quora t...,why do not many people posting questions on qu...


In [8]:
q1 = df["q1_classic"].astype(str).tolist()
q2 = df["q2_classic"].astype(str).tolist()

bow = build_bow(q1, q2, max_features=50000, ngram_range=(1, 2), min_df=2)

bow_cos = cosine_sim_sparse(bow.q1, bow.q2)
bow_euc = l2_distance_sparse(bow.q1, bow.q2)
bow_man = l1_distance_sparse(bow.q1, bow.q2)

bow_sim = build_similarity_frame(df["id"], cosine=bow_cos, euclidean=bow_euc, manhattan=bow_man, prefix="bow")
bow_sim.head()

,id,bow_cosine,bow_euclidean,bow_manhattan
0,332278,0.000000,3.464102,10
1,196656,0.816497,1.414214,2
2,113125,0.267261,2.645751,7
3,266232,0.632456,2.236068,5
4,122738,0.347833,5.291503,28


In [9]:
tfidf = build_tfidf(q1, q2, max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)

tfidf_cos = cosine_sim_sparse(tfidf.q1, tfidf.q2)
tfidf_euc = l2_distance_sparse(tfidf.q1, tfidf.q2)
tfidf_man = l1_distance_sparse(tfidf.q1, tfidf.q2)

tfidf_sim = build_similarity_frame(df["id"], cosine=tfidf_cos, euclidean=tfidf_euc, manhattan=tfidf_man, prefix="tfidf")
tfidf_sim.head()

,id,tfidf_cosine,tfidf_euclidean,tfidf_manhattan
0,332278,0.000000,1.414214,4.068060
1,196656,0.847734,0.551844,0.785760
2,113125,0.219413,1.249470,3.375859
3,266232,0.396429,1.098700,2.709021
4,122738,0.176252,1.283548,6.882932


In [10]:
q1n = df["q1_norm"].astype(str).tolist()
q2n = df["q2_norm"].astype(str).tolist()

g_q1, g_q2 = build_glove_avg(q1n, q2n, glove_name="glove-wiki-gigaword-100")

glove_cos = cosine_sim_dense(g_q1, g_q2)
glove_euc = l2_distance_dense(g_q1, g_q2)
glove_man = l1_distance_dense(g_q1, g_q2)

glove_sim = build_similarity_frame(df["id"], cosine=glove_cos, euclidean=glove_euc, manhattan=glove_man, prefix="glove100")
glove_sim.head()

[=========-----------------------------------------] 18.3% 23.4/128.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[=====================-----------------------------] 43.6% 55.8/128.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[=================================-----------------] 67.4% 86.3/128.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==================================================] 100.0% 128.1/128.1MB downloaded


,id,glove100_cosine,glove100_euclidean,glove100_manhattan
0,332278,0.853573,2.294052,18.518164
1,196656,0.959000,1.294360,10.086360
2,113125,0.898489,1.951799,15.787617
3,266232,0.973897,1.019864,8.242043
4,122738,0.974563,0.969963,7.853030


In [11]:
use_word2vec = False

In [12]:
if use_word2vec:
    w_q1, w_q2 = build_word2vec_avg(q1n, q2n, w2v_name="word2vec-google-news-300")

    w2v_cos = cosine_sim_dense(w_q1, w_q2)
    w2v_euc = l2_distance_dense(w_q1, w_q2)
    w2v_man = l1_distance_dense(w_q1, w_q2)

    w2v_sim = build_similarity_frame(df["id"], cosine=w2v_cos, euclidean=w2v_euc, manhattan=w2v_man, prefix="w2v300")
else:
    w2v_sim = None
w2v_sim.head() if w2v_sim is not None else None

In [13]:
bert_model_name = "sentence-transformers/all-MiniLM-L6-v2"
b_q1, b_q2 = build_bert_embeddings(q1n, q2n, model_name=bert_model_name, batch_size=128, normalize=False)

bert_cos = cosine_sim_dense(b_q1, b_q2)
bert_euc = l2_distance_dense(b_q1, b_q2)
bert_man = l1_distance_dense(b_q1, b_q2)

bert_sim = build_similarity_frame(df["id"], cosine=bert_cos, euclidean=bert_euc, manhattan=bert_man, prefix="bert_minilm")
bert_sim.head()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Dmity\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dmity\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2527 [00:00<?, ?it/s]

Batches:   0%|          | 0/2527 [00:00<?, ?it/s]

,id,bert_minilm_cosine,bert_minilm_euclidean,bert_minilm_manhattan
0,332278,0.154059,1.300724,19.766121
1,196656,0.869581,0.510723,7.993021
2,113125,0.169642,1.288688,19.727543
3,266232,0.787774,0.651500,10.135122
4,122738,0.684328,0.794572,12.025077


In [14]:
out = df[["id", "is_duplicate"]].copy() if "is_duplicate" in df.columns else df[["id"]].copy()

out = out.merge(bow_sim, on="id", how="left")
out = out.merge(tfidf_sim, on="id", how="left")
out = out.merge(glove_sim, on="id", how="left")
out = out.merge(bert_sim, on="id", how="left")

if w2v_sim is not None:
    out = out.merge(w2v_sim, on="id", how="left")

OUT_PATH = PROCESSED_DIR / "pair_similarity_features.csv"
out.to_csv(OUT_PATH, index=False)

OUT_PATH, out.shape

(WindowsPath('D:/Git/Quora-project/data/processed/pair_similarity_features.csv'),
 (323432, 14))